# Consolidated AA Plots

This notebook consolidates publication-ready visualizations for Dog, Age, and Sample Type datasets in one place.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from archetypal_analysis.data.loaders import ArchetypalResultLoader
from archetypal_analysis.analysis.metrics import AnalysisMetrics
from archetypal_analysis.visualization.simplex import SimplexPlotter
from archetypal_analysis.visualization.stacked_bar import StackedBarPlotter
from archetypal_analysis.visualization.composition import CompositionAnalyzer
from archetypal_analysis.utils.metadata import extract_metadata


## Plotters

In [ ]:
ROOT = '.'

def set_paper_style():
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size': 12,
        'axes.titlesize': 14,
        'axes.labelsize': 12,
        'legend.fontsize': 10,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'axes.linewidth': 0.8,
        'grid.alpha': 0.25
    })

def read_table_auto(path):
    if path.endswith('.tsv'):
        return pd.read_table(path, index_col=0)
    return pd.read_csv(path, index_col=0)

def result_path(result_dir, k):
    return os.path.join(result_dir, f'otu_AA_K{k}_1.0_None_jensenshannon.json')

def build_uniform_archetype_colors(k):
    return {f'Archetype {i+1}': (i + 1) / max(k, 1) for i in range(k)}

def reorder_similarity_and_archetypes(otu_aa, aa_archetypes, order_cols=None):
    if order_cols is None:
        return otu_aa.copy(), np.array(aa_archetypes)

    reordered = otu_aa[order_cols].copy()
    renamed_cols = [f'Archetype {i+1}' for i in range(len(order_cols))]
    reordered.columns = renamed_cols

    idx = [int(c.split()[-1]) - 1 for c in order_cols]
    aa_reordered = np.array(aa_archetypes)[idx, :]
    return reordered, aa_reordered

def load_dataset_bundle(cfg):
    loader = ArchetypalResultLoader()

    otu = read_table_auto(cfg['otu_file'])
    meta = read_table_auto(cfg['meta_file'])

    projects = None
    tags = None
    if cfg.get('projects_file'):
        projects = pd.read_csv(cfg['projects_file'], index_col=0)
    if cfg.get('tags_file'):
        tags = pd.read_table(cfg['tags_file'], index_col=0)

    otu_aa, aa_labels, aa_archetypes, info = loader.load_parsed_result(result_path(cfg['result_dir'], cfg['main_k']))
    otu_aa, aa_archetypes = reorder_similarity_and_archetypes(otu_aa, aa_archetypes, cfg.get('archetype_order'))

    metadata = extract_metadata(otu_aa.index, tags, meta, projects, [cfg['metadata_col']], verbose=False)

    return {
        'otu': otu,
        'meta': meta,
        'projects': projects,
        'tags': tags,
        'otu_aa': otu_aa,
        'aa_labels': aa_labels,
        'aa_archetypes': aa_archetypes,
        'info': info,
        'metadata': metadata
    }

def plot_pca_3d(otu, aa_archetypes, metadata, cfg):
    arch_df = pd.DataFrame(aa_archetypes, columns=otu.columns, index=[f'Archetype {i+1}' for i in range(len(aa_archetypes))])
    combined = pd.concat([otu, arch_df], axis=0)

    X = np.log1p(combined.to_numpy())
    X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
    pca = PCA(n_components=3, random_state=42)
    pcs = pca.fit_transform(X)

    pca_df = pd.DataFrame(pcs, index=combined.index, columns=['PC1', 'PC2', 'PC3'])
    sample_idx = otu.index.intersection(metadata.index)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    if cfg['legend_type'] == 'continuous':
        vals = pd.to_numeric(metadata.loc[sample_idx, cfg['metadata_col']], errors='coerce')
        scatter = ax.scatter(
            pca_df.loc[sample_idx, 'PC1'],
            pca_df.loc[sample_idx, 'PC2'],
            pca_df.loc[sample_idx, 'PC3'],
            c=vals,
            cmap=cfg['cmap'],
            s=20,
            alpha=0.7
        )
        cbar = fig.colorbar(scatter, ax=ax, pad=0.08, shrink=0.65)
        cbar.set_label(cfg['metadata_col'])
    else:
        labels = metadata.loc[sample_idx, cfg['metadata_col']].fillna('unknown').astype(str)
        unique_labels = sorted(labels.unique())
        label_to_idx = {lab: i for i, lab in enumerate(unique_labels)}
        color_vals = labels.map(label_to_idx).to_numpy()
        cmap_obj = cm.get_cmap(cfg['cmap'], max(len(unique_labels), 1))

        ax.scatter(
            pca_df.loc[sample_idx, 'PC1'],
            pca_df.loc[sample_idx, 'PC2'],
            pca_df.loc[sample_idx, 'PC3'],
            c=color_vals,
            cmap=cmap_obj,
            s=20,
            alpha=0.7
        )

        legend_handles = [
            Line2D([0], [0], marker='o', linestyle='', color=cmap_obj(i), label=lab, markersize=6)
            for lab, i in label_to_idx.items()
        ]
        ax.legend(handles=legend_handles, title=cfg['metadata_col'], loc='upper left', bbox_to_anchor=(1.02, 1.0))

    arch_coords = pca_df.loc[arch_df.index]
    arch_cmap = cm.get_cmap(cfg['cmap'])
    arch_color_map = cfg['archetype_colors']

    for arch_name, row in arch_coords.iterrows():
        ax.scatter(
            row['PC1'], row['PC2'], row['PC3'],
            marker='X', s=260,
            color=arch_cmap(arch_color_map.get(arch_name, 0.5)),
            edgecolor='black', linewidth=0.8
        )
        ax.text(row['PC1'], row['PC2'], row['PC3'], arch_name, fontsize=9)

    evr = pca.explained_variance_ratio_
    ax.set_title(f"{cfg['title']}: PCA (PC1 vs PC2 vs PC3)")
    ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
    ax.set_zlabel(f'PC3 ({evr[2]*100:.1f}%)')
    plt.tight_layout()
    plt.show()

def plot_scores_vs_k(cfg, metadata, loader):
    k_values = cfg['k_values']
    results = loader.load_multiple_results(k_values=k_values, directory=cfg['result_dir'])

    metrics = AnalysisMetrics()
    comp = metrics.compare_archetype_quality(
        results_dict=results,
        metadata=metadata,
        metadata_column=cfg['metadata_col'],
        discretize=cfg.get('discretize_scores', False),
        buckets=cfg.get('score_buckets', None),
        metrics=['silhouette', 'davies_bouldin']
    )

    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax2 = ax1.twinx()

    ax1.plot(comp['k'], comp['silhouette'], color='#1f77b4', marker='o', linewidth=2, label='Silhouette')
    ax2.plot(comp['k'], comp['davies_bouldin'], color='#d62728', marker='s', linewidth=2, label='Davies-Bouldin')

    ax1.set_xlabel('Number of Archetypes (K)')
    ax1.set_ylabel('Silhouette Score', color='#1f77b4')
    ax2.set_ylabel('Davies-Bouldin Score', color='#d62728')
    ax1.set_title(f"{cfg['title']}: Davies-Bouldin + Silhouette vs Number of Archetypes")
    ax1.grid(True, alpha=0.25)

    lines = ax1.get_lines() + ax2.get_lines()
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='best', frameon=True)
    plt.tight_layout()
    plt.show()

def plot_simplex_panel(otu_aa, metadata, cfg):
    simplex_plotter = SimplexPlotter()
    simplex_plotter.plot_simplex(
        otu_aa=otu_aa,
        metadata=metadata,
        filter_cols=[cfg['metadata_col']],
        cmap_name=cfg['cmap'],
        legend_type=cfg['legend_type'],
        remove_nan=True,
        alpha=0.75,
        s=30,
        figsize=(9, 8),
        remove_values=cfg.get('remove_values', {}),
        legend_fontsize=10
    )
    plt.title(f"{cfg['title']}: Archetypal Analysis Simplex")
    plt.tight_layout()
    plt.show()

def plot_similarity_stacked(otu_aa, info, metadata, cfg):
    stacked = StackedBarPlotter()
    stacked.plot_metadata_stacked_bars(
        similarity_degree=otu_aa,
        metadata=metadata,
        perms=info.get('perms', None),
        metadata_column=cfg['metadata_col'],
        discretize=cfg.get('discretize_stacked', False),
        buckets=cfg.get('stacked_buckets', None),
        cmap_name=cfg['cmap'],
        vertex_colors=cfg['archetype_colors'],
        figsize=(12, 4),
        alpha=0.9
    )
    plt.title(f"{cfg['title']}: Similarity Degree Stacked Bar")
    plt.tight_layout()
    plt.show()

def plot_taxa_contribution(otu, aa_archetypes, cfg):
    comp = CompositionAnalyzer()
    top_taxa = comp.analyze_top_taxa(
        archetypes=np.array(aa_archetypes),
        feature_names=otu.columns,
        num_taxa=cfg.get('num_taxa', 5)
    )

    comp.plot_stacked_bar_chart(
        aa_archetypes=np.array(aa_archetypes),
        feature_names=otu.columns,
        top_taxa_archetypes=top_taxa,
        figsize=(11, 6),
        sort_by_contribution=True,
        simplify_names=True,
        max_name_length=18
    )
    plt.title(f"{cfg['title']}: Taxa Contribution Bar Plot")
    plt.tight_layout()
    plt.show()

def run_all_plots_for_dataset(cfg):
    set_paper_style()
    loader = ArchetypalResultLoader()
    bundle = load_dataset_bundle(cfg)

    plot_pca_3d(bundle['otu'], bundle['aa_archetypes'], bundle['metadata'], cfg)
    plot_scores_vs_k(cfg, bundle['metadata'], loader)
    plot_simplex_panel(bundle['otu_aa'], bundle['metadata'], cfg)
    plot_similarity_stacked(bundle['otu_aa'], bundle['info'], bundle['metadata'], cfg)
    plot_taxa_contribution(bundle['otu'], bundle['aa_archetypes'], cfg)

AGE_BUCKETS = [(0, 6), (6, 12), (12, 18), (18, 24), (24, 30), (30, 36), (36, 42), (42, 48), (48, 54), (54, 73)]

DATASET_CONFIGS = {
    'dog': {
        'title': 'Dog (Lifestyle200)',
        'result_dir': os.path.join(ROOT, 'results', 'Dog', 'norm_lifestyle200'),
        'main_k': 8,
        'k_values': list(range(3, 14)),
        'otu_file': os.path.join(ROOT, 'datasets', 'tsv', 'normalized', 'dog_otu_normalized_Lifestyle.csv'),
        'meta_file': os.path.join(ROOT, 'datasets', 'tsv', 'dog_sam.tsv'),
        'projects_file': None,
        'tags_file': None,
        'metadata_col': 'Lifestyle',
        'legend_type': 'discrete',
        'cmap': 'Set1',
        'archetype_order': None,
        'archetype_colors': build_uniform_archetype_colors(8),
        'remove_values': {'Lifestyle': 'unknown'},
        'num_taxa': 5
    },
    'age': {
        'title': 'Hadza (Age1k)',
        'result_dir': os.path.join(ROOT, 'results', 'Hadza', 'norm_qc_age1k'),
        'main_k': 8,
        'k_values': list(range(3, 14)),
        'otu_file': os.path.join(ROOT, 'datasets', 'tsv', 'normalized', 'hadza_otu_normalized_age.csv'),
        'meta_file': os.path.join(ROOT, 'datasets', 'tsv', 'hadza_sam.tsv'),
        'projects_file': None,
        'tags_file': None,
        'metadata_col': 'age_months',
        'legend_type': 'continuous',
        'cmap': 'viridis',
        'archetype_order': ['Archetype 4', 'Archetype 5', 'Archetype 3', 'Archetype 2', 'Archetype 6', 'Archetype 7', 'Archetype 8', 'Archetype 1'],
        'archetype_colors': build_uniform_archetype_colors(8),
        'discretize_scores': True,
        'score_buckets': AGE_BUCKETS,
        'discretize_stacked': True,
        'stacked_buckets': AGE_BUCKETS,
        'remove_values': {'age_months': 'unknown'},
        'num_taxa': 5
    },
    'sample_type': {
        'title': 'Compendium (Sample Type400)',
        'result_dir': os.path.join(ROOT, 'results', 'Compendium_N3_N13_NoPC_NoGlom', 'Norm_pp_100iter_QC-Sample_type400'),
        'main_k': 7,
        'k_values': list(range(3, 14)),
        'otu_file': os.path.join(ROOT, 'datasets', 'compendium', 'normalized', 'compendium_Norm_QC-sample_type400.csv'),
        'meta_file': os.path.join(ROOT, 'datasets', 'compendium', 'sample_metadata.tsv'),
        'projects_file': os.path.join(ROOT, 'datasets', 'compendium', 'projects.csv'),
        'tags_file': os.path.join(ROOT, 'datasets', 'compendium', 'tags.tsv'),
        'metadata_col': 'sample_type',
        'legend_type': 'discrete',
        'cmap': 'tab20',
        'archetype_order': ['Archetype 1', 'Archetype 3', 'Archetype 2', 'Archetype 5', 'Archetype 6', 'Archetype 4', 'Archetype 7'],
        'archetype_colors': {'Archetype 1': 6/20, 'Archetype 2': 0/20, 'Archetype 3': 4/20, 'Archetype 4': 12/20, 'Archetype 5': 8/20, 'Archetype 6': 18/20, 'Archetype 7': 16/20},
        'remove_values': {'sample_type': 'unknown'},
        'num_taxa': 5
    }
}

print("Plotters loaded. Use run_all_plots_for_dataset(DATASET_CONFIGS['dog' | 'age' | 'sample_type'])")


## Dog Dataset Results

In [ ]:
run_all_plots_for_dataset(DATASET_CONFIGS['dog'])

## Age Dataset Results

In [ ]:
run_all_plots_for_dataset(DATASET_CONFIGS['age'])

## Sample Type Dataset Results

In [ ]:
run_all_plots_for_dataset(DATASET_CONFIGS['sample_type'])